In [1]:
import os
import torch 
from torchvision import transforms
from torch.utils.data import DataLoader
from dataset import build_dataset
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from models import SimpleCNNClassifier
from tqdm import tqdm
from shared import ModelCheckpoint
import pandas as pd

### Configurations

Run

In [2]:
run_name = "sh1"

Dataset

In [3]:
dataset_path = "/home/furkan/projects/cpak/preprocessing/knee-preop-prepped"
csv_path = "fibula.csv"
seed = 42
test_size = 0.2

Dataloader

In [4]:
batch_size = 16
num_workers = 2

Train

In [5]:
num_epoch = 10
lr = 0.001
checkpoint_frequency = 1
resume = False

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Dataset

In [7]:
train_transform = transforms.Compose([
	transforms.ToTensor(),
	transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # normalization for RGB
])

test_transform = transforms.Compose([
	transforms.ToTensor(),
	transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # normalization for RGB
])

In [8]:
trainset, testset = build_dataset(
	csv_path=csv_path,
	root_dir=dataset_path,
	seed=42,
	test_size=test_size,
	train_transform=train_transform,
	test_transform=test_transform
)

## Dataloader

In [9]:
train_loader = DataLoader(
	trainset,
	batch_size=batch_size,
	shuffle=True,
	num_workers=num_workers
)

test_loader = DataLoader(
	testset,
	batch_size=batch_size,
	shuffle=True,
	num_workers=num_workers
)

## Train

In [10]:
model = SimpleCNNClassifier(batch_norm=True).to(device)
criterion = CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=lr)

In [11]:
os.makedirs(f"out/models/{run_name}", exist_ok=True)
print(f"Training starts with {len(trainset)} images")

for epoch in range(num_epoch):
	model.train()
	running_loss = 0.0
	loop = tqdm(
		train_loader, 
		desc=f"Epoch [{epoch+1}/{num_epoch}]", 
		leave=False
	)
	
	for data in loop:

		inputs = data["image"].to(device)
		labels = data["label"].to(device)

		optimizer.zero_grad()

		preds = model(inputs)

		loss = criterion(preds, labels)

		loss.backward()

		optimizer.step()
	
		running_loss += loss.item()

		loop.set_postfix(batch_loss=loss.item())

	# Calculate average loss over the entire epoch
	epoch_loss = running_loss / len(train_loader)

	# Print the final summary for the epoch
	print(f"⏩ Epoch {epoch+1} Completed | Average Loss: {epoch_loss:.4f}")

	# dont save when epoch=0 but save when epoch=999 (never reaches 1000)
	if (epoch+1) % checkpoint_frequency == 0:
		checkpoint:ModelCheckpoint = {
			"model_state":model.state_dict(),
			"optimizer_state":optimizer.state_dict(),
			"epoch":epoch+1,
			"epoch_loss":epoch_loss
		}

		checkpoint_file_name = f"out/models/{run_name}/ep{epoch+1}.pt"

		torch.save(checkpoint, checkpoint_file_name)

		print(f"Successfully saved epoch {epoch+1} to {checkpoint_file_name}.")

		# === evaluation ===
		model.eval()

		inference_records = []
		with torch.no_grad():
			for split in ["train", "test"]:
				loader = train_loader if split == "train" else test_loader
				for batch in loader:

					inputs = batch["image"].to(device)
					labels = batch["label"].to(device)
					image_names = batch["name"] 

					logits = model(inputs)
					
					_, preds = torch.max(logits, 1)

					flat_preds = preds.cpu().tolist()
					flat_labels = labels.cpu().tolist()

					for name, p, l, logit in zip(image_names, flat_preds, flat_labels, logits):
						inference_records.append({
							"epoch": epoch + 1,
							"split": split,
							"image_name": name,
							"prediction": p,
							"true_label": l,
							"logits": logit
						})
						
		results_df = pd.DataFrame(inference_records)
		csv_output_path = f"out/models/{run_name}/ep{epoch+1}.csv"
		results_df.to_csv(csv_output_path, index=False)

Training starts with 154 images


⏩ Epoch 1 Completed | Average Loss: 0.4574
Successfully saved epoch 1 to out/models/sh1/ep1.pt.


⏩ Epoch 2 Completed | Average Loss: 0.2960
Successfully saved epoch 2 to out/models/sh1/ep2.pt.


⏩ Epoch 3 Completed | Average Loss: 0.2370
Successfully saved epoch 3 to out/models/sh1/ep3.pt.


⏩ Epoch 4 Completed | Average Loss: 0.2103
Successfully saved epoch 4 to out/models/sh1/ep4.pt.


...

⏩ Epoch 6 Completed | Average Loss: 0.1618
Successfully saved epoch 6 to out/models/sh1/ep6.pt.


⏩ Epoch 7 Completed | Average Loss: 0.1288
Successfully saved epoch 7 to out/models/sh1/ep7.pt.


⏩ Epoch 8 Completed | Average Loss: 0.1288
Successfully saved epoch 8 to out/models/sh1/ep8.pt.


⏩ Epoch 9 Completed | Average Loss: 0.0989
Successfully saved epoch 9 to out/models/sh1/ep9.pt.


⏩ Epoch 10 Completed | Average Loss: 0.0921
Successfully saved epoch 10 to out/models/sh1/ep10.pt.
